In [ ]:
import sys

sys.path.append("..")          # notebooks/ -> repo root, so `import src...` works
from pyspark.sql import functions as F

from src.paths import BRONZE, SILVER
from src.silver import END, SCOPED_FAMILIES, SCOPED_STORES, START, build_silver, scope
from src.spark import get_spark

spark = get_spark()
spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/22 10:31:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
b = {n: spark.read.parquet(str(BRONZE / n)) for n in ["sales", "stores", "oil", "holidays", "transactions"]}

In [3]:
silver = build_silver(b["sales"], b["stores"], b["oil"], b["holidays"], b["transactions"], START, END)
n = silver.count()
print(n)
assert n == 54 * 33 * 1688
silver.write.mode("overwrite").partitionBy("store_nbr").parquet(str(SILVER / "sales_daily"))

26/09/22 10:31:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/22 10:31:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


3008016


26/09/22 10:31:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/22 10:31:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/22 10:31:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/22 10:31:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/22 10:31:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/22 10:31:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/22 1

In [4]:
sd = spark.read.parquet(str(SILVER / "sales_daily"))
sd.filter(F.col("store_nbr") == 44).explain()

== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [date#231,family#232,sales#233,onpromotion#234,city#235,state#236,type#237,cluster#238,oil_price#239,national_holiday#240,any_holiday#241,transactions#242,store_nbr#243] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/Users/levente/Official/Projects/Grocery-Demand-Intelligence/data..., PartitionFilters: [isnotnull(store_nbr#243), (store_nbr#243 = 44)], PushedFilters: [], ReadSchema: struct<date:date,family:string,sales:double,onpromotion:int,city:string,state:string,type:string,...




In [5]:
scoped = scope(sd, SCOPED_STORES, SCOPED_FAMILIES)
assert scoped.count() == 12 * 8 * 1688
scoped.write.mode("overwrite").partitionBy("store_nbr").parquet(str(SILVER / "sales_daily_scoped"))

In [6]:
assert scoped.groupBy("store_nbr", "family", "date").count().filter("count > 1").count() == 0
scoped.filter(F.col("oil_price").isNull()).select("date").distinct().show()

+----------+
|      date|
+----------+
|2013-01-01|
+----------+

